# Do rim/interior "gates" explain boundary occupation of generalised Langton's ants?

**For a colleague who has not seen this work.** Rules are strings over {L,R}; the walker's *boundary occupation* is the share of
steps spent on a visited cell with an unvisited neighbour. Earlier experiments (see `notes/kappa-mechanisms-2026-10-15.md`)
could not explain how this varies across rules. Hypothesis tested here: a few directional crossings between the active rim
(visited depth <= 4 from the boundary) and the interior carry the information. **Stage 1** asks whether two gate summaries
improve held-out prediction over a baseline. Stage 2 (interventions) was *not* attempted.

**Answer in one line:** no improvement at either area band, but the verdict is *UNINFORMATIVE*: the baseline already explains
88-95% of the variance (no headroom at band 1) and one gate feature was unreliable at band 0.

## Assumptions and design (frozen before the confirmation run; full text in README.md, 'REGISTRATION')
- 24 rules, 6 per {crust, uniform} x {stationary, nonstationary} cell, from existing labels; fresh random-patch starts.
- Window: departures with visited area in [A, A + A/16), A = 32768 and 131072. Predictor seeds (64/rule) and target seeds (32/rule) are disjoint.
- Target: ln(boundary steps / steps) from target seeds. Baseline B0: 4 features from predictor seeds that exclude boundary occupancy itself.
- Gate features: g1 = excess concentration of crossings in a few (tile, heading, kind) bins vs an exposure-matched Poisson null; g2 = spread over exit headings of ln mean excursion length.
- Ridge (lambda 1), leave-one-rule-out; pass = DeltaR2 >= 0.10, bootstrap lower bound > 0, permutation p <= 0.05, at both bands; gates: coverage, split-seed reliability >= 0.7, headroom R2(B0) <= 0.90.

In [1]:
import json, hashlib, subprocess
from pathlib import Path
import polars as pl
EXP = Path.cwd()
while not (EXP / 'results').is_dir() and EXP != EXP.parent:
    EXP = EXP.parent if (EXP / 'README.md').exists() is False else EXP
R = EXP / 'results'
if not R.is_dir():
    R = Path('experiments/archive/2026-10-kappa-return-gates/results')
files = ['cohort.csv','pilot_runs.csv','confirm_runs.csv','confirm_analysis.json','pilot_analysis.json','controls_analysis.json','g1_null_calibration.json','determinism.json','sensitivity_g1_nan.json']
prov = {f: hashlib.sha256((R/f).read_bytes()).hexdigest()[:16] for f in files}
conf = json.loads((R/'confirm_analysis.json').read_text())
print('analysis commit (base, dirty tree):', conf['provenance']['git_commit'])
print(json.dumps(prov, indent=1))

analysis commit (base, dirty tree): baac481911aa68db941e6b692be978dcfb5f08af
{
 "cohort.csv": "21eb3858b2f75131",
 "pilot_runs.csv": "7f252a0552fe770b",
 "confirm_runs.csv": "5c64ced0ca0ff85c",
 "confirm_analysis.json": "ed6ed12ba8c1d415",
 "pilot_analysis.json": "3b4ec34f19493eab",
 "controls_analysis.json": "dfc3f676b795b204",
 "g1_null_calibration.json": "86205c58d2830f81",
 "determinism.json": "7224aca19134834f",
 "sensitivity_g1_nan.json": "edc397e69a56e08f"
}


## Provenance and instrument checks

In [2]:
print('determinism:', json.loads((R/'determinism.json').read_text()))
print((R/'return_gates_tests.log').read_text().strip())
cal = json.loads((R/'g1_null_calibration.json').read_text())
print(pl.DataFrame(cal['rows']))

determinism: {'specs': 12, 'files': 36, 'one_vs_six_threads_identical': True, 'interrupted_resumed_identical': True, 'note': 'resumed copy: 4 specs deleted, 1 spec left with a stray .tmp and no .done; rerun with 3 threads; finished specs skipped'}
test result: ok. 5 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 2.02s
shape: (5, 7)
┌──────┬─────────┬───────────┬─────────┬──────────────┬────────────┬──────────────────────┐
│ bins ┆ events  ┆ null_mean ┆ null_sd ┆ planted_mean ┆ planted_sd ┆ planted_over_null_sd │
│ ---  ┆ ---     ┆ ---       ┆ ---     ┆ ---          ┆ ---        ┆ ---                  │
│ i64  ┆ i64     ┆ f64       ┆ f64     ┆ f64          ┆ f64        ┆ f64                  │
╞══════╪═════════╪═══════════╪═════════╪══════════════╪════════════╪══════════════════════╡
│ 30   ┆ 400     ┆ 0.00143   ┆ 0.02037 ┆ 0.05257      ┆ 0.10569    ┆ 2.58                 │
│ 300  ┆ 1000    ┆ -0.0012   ┆ 0.014   ┆ 0.06138      ┆ 0.039      ┆ 4.38                 │


The incremental depth map agrees with a from-scratch BFS oracle on random rules (property test), 1 and 6 threads give identical bytes, and an interrupted run resumes to identical bytes. The gate-concentration null is unbiased on synthetic uniform flow but has weak power in the sparse regime (30 bins, 400 events) that real runs partly occupy.

## Cohort

In [3]:
print(pl.read_csv(R/'cohort.csv').select('base','crust','stationary','c','g_k'))

shape: (24, 5)
┌──────────────┬───────┬────────────┬──────────┬───────────┐
│ base         ┆ crust ┆ stationary ┆ c        ┆ g_k       │
│ ---          ┆ ---   ┆ ---        ┆ ---      ┆ ---       │
│ str          ┆ bool  ┆ bool       ┆ f64      ┆ f64       │
╞══════════════╪═══════╪════════════╪══════════╪═══════════╡
│ RLLRRLLRL    ┆ true  ┆ true       ┆ 0.654188 ┆ -0.029113 │
│ RLLLLRLRRRLL ┆ true  ┆ true       ┆ 0.659089 ┆ 0.006106  │
│ RLLLLLLRLLRL ┆ true  ┆ true       ┆ 0.654129 ┆ -0.028862 │
│ RLLLRRRRRRR  ┆ true  ┆ true       ┆ 0.664075 ┆ 0.0067    │
│ RLLRRRLLLLLR ┆ true  ┆ true       ┆ 0.63661  ┆ 0.00526   │
│ …            ┆ …     ┆ …          ┆ …        ┆ …         │
│ RRLRLLRRLLLL ┆ false ┆ false      ┆ 0.441527 ┆ -0.072361 │
│ RLLRRRLLRLRR ┆ false ┆ false      ┆ 0.499999 ┆ -0.090891 │
│ RLLLRLRRRR   ┆ false ┆ false      ┆ 0.50889  ┆ -0.08133  │
│ RRLRLLLLLL   ┆ false ┆ false      ┆ 0.444082 ┆ 0.070425  │
│ RRLRLLLRLRRR ┆ false ┆ false      ┆ 0.4438   ┆ 0.051506  │
└────────

## Pilot (8 seeds/rule): reliability and the seed-count rule

In [4]:
p = json.loads((R/'pilot_analysis.json').read_text())
rows = [dict(band=b, feature=f, split_half_spearman_4v4=p['bands'][b][f]['split_half_spearman_4v4'], projected_64=p['bands'][b][f]['projected']['64']) for b in ('0','1') for f in ('g1','g2','y')]
print(pl.DataFrame(rows)); print('S_pred =', p['S_pred'])

shape: (6, 4)
┌──────┬─────────┬─────────────────────────┬──────────────┐
│ band ┆ feature ┆ split_half_spearman_4v4 ┆ projected_64 │
│ ---  ┆ ---     ┆ ---                     ┆ ---          │
│ str  ┆ str     ┆ f64                     ┆ f64          │
╞══════╪═════════╪═════════════════════════╪══════════════╡
│ 0    ┆ g1      ┆ 0.58087                 ┆ 0.956849     │
│ 0    ┆ g2      ┆ 0.147374                ┆ 0.734435     │
│ 0    ┆ y       ┆ 0.710435                ┆ 0.975159     │
│ 1    ┆ g1      ┆ 0.665217                ┆ 0.969505     │
│ 1    ┆ g2      ┆ 0.569735                ┆ 0.954927     │
│ 1    ┆ y       ┆ 0.593043                ┆ 0.958875     │
└──────┴─────────┴─────────────────────────┴──────────────┘
S_pred = 64


## Confirmation: the registered test

In [5]:
rows = []
for b, r in conf['bands'].items():
    rows.append(dict(band=int(b), rules=r['n_rules_evaluable'], R2_B0=r['r2_baseline'], R2_with_gates=r['r2_with_gates'], dR2=r['delta_r2'], ci_lo=r['delta_ci95'][0], ci_hi=r['delta_ci95'][1], perm_p=r['permutation_p'], rel_g1=r['reliability']['g1']['split_half_spearman'], rel_g2=r['reliability']['g2']['split_half_spearman']))
print(pl.DataFrame(rows)); print(conf['gates']); print(conf['verdict'])

shape: (2, 10)
┌──────┬───────┬──────────┬───────────────┬───┬──────────┬──────────┬──────────┬──────────┐
│ band ┆ rules ┆ R2_B0    ┆ R2_with_gates ┆ … ┆ ci_hi    ┆ perm_p   ┆ rel_g1   ┆ rel_g2   │
│ ---  ┆ ---   ┆ ---      ┆ ---           ┆   ┆ ---      ┆ ---      ┆ ---      ┆ ---      │
│ i64  ┆ i64   ┆ f64      ┆ f64           ┆   ┆ f64      ┆ f64      ┆ f64      ┆ f64      │
╞══════╪═══════╪══════════╪═══════════════╪═══╪══════════╪══════════╪══════════╪══════════╡
│ 0    ┆ 22    ┆ 0.881662 ┆ 0.842208      ┆ … ┆ 0.005828 ┆ 0.846307 ┆ 0.917561 ┆ 0.564348 │
│ 1    ┆ 24    ┆ 0.945774 ┆ 0.945364      ┆ … ┆ 0.030987 ┆ 0.05988  ┆ 0.928696 ┆ 0.815652 │
└──────┴───────┴──────────┴───────────────┴───┴──────────┴──────────┴──────────┴──────────┘
{'0': {'coverage': True, 'reliability': False, 'headroom': True}, '1': {'coverage': True, 'reliability': True, 'headroom': False}}
UNINFORMATIVE: at least one gate failed (band 0: ['reliability'], band 1: ['headroom']); G1 criterion would have FAILE

**Reading.** DeltaR2 is zero or negative with upper bounds (+0.006, +0.031) far below the +0.10 criterion, so the prediction is not supported.
But the test is mostly decided by the baseline (R2 0.88 and 0.95), and g2 at band 0 fails its reliability gate, so this is an
uninformative rather than a clean rejection. It does **not** show gates are causally irrelevant.

## Exploratory checks (not gating)

In [6]:
for b, r in conf['bands'].items():
    print('band', b, r['exploratory'])
s = json.loads((R/'sensitivity_g1_nan.json').read_text())
print('sensitivity (3 NaN-g1 runs ignored, 24 rules at band 0):', {k: s['0'][k] for k in ('n_rules_evaluable','delta_r2','delta_ci95','permutation_p')})
print(pl.DataFrame(json.loads((R/'controls_analysis.json').read_text())).select('walker','band','g1_mean','boundary_occupancy_mean','hole_boundary_fraction'))

band 0 {'g1_only_delta_r2': -0.017567822655202847, 'g2_only_delta_r2': -0.03490082111939552, 'oracle_lnq_delta_r2': 0.0011544946531061306, 'rule_strata_in_evaluable': [{'crust': False, 'stationary': False, 'len': 5}, {'crust': True, 'stationary': True, 'len': 5}, {'crust': False, 'stationary': True, 'len': 6}, {'crust': True, 'stationary': False, 'len': 6}], 'spearman_y_g1': -0.033314511575381144, 'spearman_y_g2': 0.3800112930547713}
band 1 {'g1_only_delta_r2': 0.012295980216782354, 'g2_only_delta_r2': -0.008131724255140527, 'oracle_lnq_delta_r2': 0.01754084595624461, 'rule_strata_in_evaluable': [{'crust': False, 'stationary': True, 'len': 6}, {'crust': False, 'stationary': False, 'len': 6}, {'crust': True, 'stationary': False, 'len': 6}, {'crust': True, 'stationary': True, 'len': 6}], 'spearman_y_g1': 0.3608695652173913, 'spearman_y_g2': 0.3652173913043478}
sensitivity (3 NaN-g1 runs ignored, 24 rules at band 0): {'n_rules_evaluable': 24, 'delta_r2': -0.029084591470921684, 'delta_ci95

## Caveats
n = 24 rules; labels come from earlier pooled summaries; bootstrap omits training-fold uncertainty; g1's absolute level is not
interpretable (rotor and random walk give -0.04 / -0.01 rather than 0); colour phase and other tile/depth choices were not tested;
three runs had a NaN null (reported, registered result unchanged). The depth-profile baseline is close to an accounting identity
with the target, which is why headroom vanished.

## Next steps
Re-test gates only against a baseline built from quantities that precede the window or from independent mechanism features;
keep the permutation criterion. G2 (matched gate vs non-gate edits from forked snapshots) needs its own registration and is
unrun. See `notes/open-questions-hypotheses.md`.